In [96]:
from dotenv import load_dotenv
import os
from fredapi import Fred
import pandas as pd
import requests
import yfinance as yf

In [97]:
load_dotenv()

FRED_API_KEY = os.getenv("FRED_API_KEY")
fred = Fred(api_key=FRED_API_KEY)

Celda 3 — Extract: Empresas S&P 500

In [98]:
# Celda 3 — Extract: Empresas S&P 500
from io import StringIO

def extract_empresas():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    
    df = pd.read_html(StringIO(response.text))[0]
    df = df.rename(columns={
        "Symbol":            "ticker",
        "Security":          "nombre",
        "GICS Sector":       "sector",
        "GICS Sub-Industry": "industria",
        "Date added":        "fecha_inclusion"
    })
    df["ticker"] = df["ticker"].str.replace(".", "-", regex=False)
    return df[["ticker", "nombre", "sector", "industria", "fecha_inclusion"]]

df_empresas_raw = extract_empresas()



In [99]:
df_empresas_raw

,ticker,nombre,sector,industria,fecha_inclusion
0,MMM,3M,Industrials,Industrial Conglomerates,1957-03-04
1,AOS,A. O. Smith,Industrials,Building Products,2017-07-26
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1957-03-04
3,ABBV,AbbVie,Health Care,Biotechnology,2012-12-31
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,2011-07-06
...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,2011-11-01
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,1997-10-06
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,2019-12-23
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,2001-08-07


In [81]:
def transform_empresas(df):
    df = df.copy()
    df["ticker"]          = df["ticker"].str.strip()
    df["nombre"]          = df["nombre"].str.strip()
    df["sector"]          = df["sector"].str.strip()
    df["industria"]       = df["industria"].str.strip()
    df["fecha_inclusion"] = pd.to_datetime(df["fecha_inclusion"], errors="coerce").dt.date
    df["pais"]            = "US"
    df["moneda"]          = "USD"
    df["activo"]          = True
    return df.reset_index(drop=True)

df_empresas = transform_empresas(df_empresas_raw)
df_empresas.head().set_index("ticker")




,nombre,sector,industria,fecha_inclusion,pais,moneda,activo
ticker,,,,,,,
MMM,3M,Industrials,Industrial Conglomerates,1957-03-04,US,USD,True
AOS,A. O. Smith,Industrials,Building Products,2017-07-26,US,USD,True
ABT,Abbott Laboratories,Health Care,Health Care Equipment,1957-03-04,US,USD,True
ABBV,AbbVie,Health Care,Biotechnology,2012-12-31,US,USD,True
ACN,Accenture,Information Technology,IT Consulting & Other Services,2011-07-06,US,USD,True


Extract: Precios históricos:




In [82]:
def extract_precios(tickers, periodo="10y"):
    datos = yf.download(
        tickers,
        period=periodo,
        auto_adjust=False,
        progress=False
    )
    
    registros = []
    for ticker in tickers:
        try:
            df_t = datos.xs(ticker, level="Ticker", axis=1).copy()
            df_t = df_t.dropna(how="all")
            df_t = df_t.reset_index()
            df_t["ticker"] = ticker
            df_t = df_t.rename(columns={
                "Date":      "fecha",
                "Open":      "open",
                "High":      "high",
                "Low":       "low",
                "Close":     "close",
                "Adj Close": "adj_close",
                "Volume":    "volume"
            })
            registros.append(df_t[["ticker","fecha","open","high","low","close","adj_close","volume"]])
        except Exception as e:
            print(f"Error con {ticker}: {e}")
            continue
    
    return pd.concat(registros, ignore_index=True)


#Creaciojn Data
tickers_prueba = df_empresas["ticker"].tolist()
df_precios_raw = extract_precios(tickers_prueba)


# Redondeo a dos decimasles
cols_precio = ["open", "high", "low", "close", "adj_close"]
df_precios_raw[cols_precio] = df_precios_raw[cols_precio].round(2)



In [83]:
#Cambio de dtype la columna de volumen a int
df_precios_raw["volume"] = df_precios_raw["volume"].astype("Int64") 

In [84]:
# Cambiamos el tipo de datos (Ticker) a category para ahorrar memoria
df_precios_raw["ticker"] = df_precios_raw["ticker"].astype("category")



In [90]:
df_precios_raw

Price,ticker,fecha,open,high,low,close,adj_close,volume
0,MMM,2016-03-03,133.29,133.52,132.53,133.26,96.09,2303496
1,MMM,2016-03-04,133.33,134.02,132.88,133.84,96.50,2112495
2,MMM,2016-03-07,133.76,134.26,132.25,134.26,96.80,2407309
3,MMM,2016-03-08,133.28,134.29,133.24,133.95,96.58,2416040
4,MMM,2016-03-09,134.20,134.43,133.38,133.85,96.51,2252068
...,...,...,...,...,...,...,...,...
1229683,ZTS,2026-02-25,128.70,129.78,127.02,128.95,128.95,3024100
1229684,ZTS,2026-02-26,129.10,131.34,128.44,129.76,129.76,3637500
1229685,ZTS,2026-02-27,128.93,132.30,128.25,131.10,131.10,5546300
1229686,ZTS,2026-03-02,130.05,130.05,127.09,128.96,128.96,3063900


Datos financieros de cada empresa

In [85]:
METRICAS_INCOME = [
    "Total Revenue", "Gross Profit", "Operating Income", "EBITDA",
    "Net Income", "Basic EPS", "Diluted EPS", "Research And Development"
]
METRICAS_BALANCE = [
    "Total Assets", "Total Liabilities Net Minority Interest",
    "Stockholders Equity", "Total Debt", "Net Debt",
    "Cash And Cash Equivalents", "Working Capital", "Net PPE"
]
METRICAS_CASHFLOW = [
    "Operating Cash Flow", "Capital Expenditure", "Free Cash Flow",
    "Cash Dividends Paid", "Depreciation And Amortization"
]

def extract_financieros(tickers):
    datos = {}
    for ticker in tickers:
        try:
            t = yf.Ticker(ticker)
            datos[ticker] = {
                "cuneta_resultados":   t.income_stmt,
                "balance_general": t.balance_sheet,
                "cashflow":      t.cashflow
            }
        except Exception as e:
            print(f"Error con {ticker}: {e}")
    return datos

tickers_todos = df_empresas["ticker"].tolist()

datos_financieros_raw = extract_financieros(tickers_todos)
print(f"Tickers extraídos: {len(datos_financieros_raw)} / {len(tickers_todos)}")




Tickers extraídos: 503 / 503


In [91]:
df_financieros = transform_financieros(
    datos_financieros_raw,
    METRICAS_INCOME, METRICAS_BALANCE, METRICAS_CASHFLOW
)
print(df_financieros.shape)


(11318, 5)


In [93]:
#Eliminacion de NaN en valor

print(f"Filas antes: {len(df_financieros)}")
print(f"NaN en valor: {df_financieros['valor'].isnull().sum()}")

df_financieros = df_financieros.dropna(subset=["valor"])

print(f"Filas después: {len(df_financieros)}")


Filas antes: 11318
NaN en valor: 1798
Filas después: 9520


In [94]:
print(df_financieros.dtypes)
print(f"\nMemoria: {df_financieros.memory_usage(deep=True).sum() / 1024:.1f} KB")


ticker             object
fecha      datetime64[ns]
tipo               object
metrica            object
valor             float64
dtype: object

Memoria: 1879.4 KB


In [95]:
df_financieros

,ticker,fecha,tipo,metrica,valor
0,MMM,2025-12-31,cashflow,Operating Cash Flow,2.306000e+09
1,MMM,2024-12-31,cashflow,Operating Cash Flow,1.819000e+09
2,MMM,2023-12-31,cashflow,Operating Cash Flow,6.680000e+09
3,MMM,2022-12-31,cashflow,Operating Cash Flow,5.591000e+09
5,MMM,2025-12-31,cashflow,Capital Expenditure,-9.100000e+08
...,...,...,...,...,...
11311,ZTS,2022-12-31,cashflow,Cash Dividends Paid,-6.110000e+08
11313,ZTS,2025-12-31,cashflow,Depreciation And Amortization,4.870000e+08
11314,ZTS,2024-12-31,cashflow,Depreciation And Amortization,4.970000e+08
11315,ZTS,2023-12-31,cashflow,Depreciation And Amortization,4.910000e+08


In [101]:
def transform_macro(fed_rate, cpi, unemployment, gdp, vix, pmi):
    indicadores = {
        "fed_rate":     (fed_rate,     "mensual"),
        "cpi":          (cpi,          "mensual"),
        "unemployment": (unemployment, "mensual"),
        "gdp":          (gdp,          "trimestral"),
        "vix":          (vix,          "diario"),
        "pmi":          (pmi,          "mensual"),
    }

    frames = []
    for nombre, (df, frecuencia) in indicadores.items():
        temp = df.copy()
        temp.index.name = "fecha"
        temp = temp.reset_index()
        temp.columns = ["fecha", "valor"]
        temp["indicador"] = nombre
        temp["frecuencia"] = frecuencia
        frames.append(temp)

    df = pd.concat(frames, ignore_index=True)
    df = df[["fecha", "indicador", "frecuencia", "valor"]]
    return df

df_macro = transform_macro(fed_rate, cpi, unemployment, gdp, vix, pmi)
print(df_macro.shape)
df_macro.head(10)



NameError: name 'fed_rate' is not defined